# 🌟 MangaTranslator — Colab Edition
**Features:** API key rotation • Model fallback chain • Auto-retry failed pages • CBZ export • Resume-friendly

> Make sure Runtime is **T4 GPU**: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# 1️⃣ Clone & Install
!git clone https://github.com/harami-boi/MangaTranslator.git 2>/dev/null; echo 'Repo ready'
%cd MangaTranslator
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade -q
!pip install -r requirements.txt -q
print('✅ All dependencies installed')

In [ ]:
# 2️⃣ Download Fonts
!mkdir -p fonts/Komika "fonts/Anime Ace"
!wget -q -O komika.zip "https://dl.dafont.com/dl/?f=komika_axis"
!wget -q -O animeace.zip "https://dl.dafont.com/dl/?f=anime_ace"
!unzip -q -j -o komika.zip "*.ttf" "*.otf" "*.TTF" "*.OTF" -d fonts/Komika/
!unzip -q -j -o animeace.zip "*.ttf" "*.otf" "*.TTF" "*.OTF" -d "fonts/Anime Ace/"
!rm -f komika.zip animeace.zip fonts/.ipynb_checkpoints -rf
print('✅ Fonts ready')

In [ ]:
# 3️⃣ Configuration — Edit these settings!

# === API KEYS (add as many as you want, rotation is automatic) ===
API_KEYS = [
    # Paste your Google AI Studio keys here (one per line)
    # "AIzaSy...",
    # "AIzaSy...",
]

# === MODEL FALLBACK CHAIN ===
# Models are tried in order. When one fails/exhausts, auto-falls to next.
MODEL_CHAIN = [
    "gemini-2.5-flash-lite",     # Fastest, cheapest, best for bulk
    "gemini-2.0-flash-lite",     # Fallback lite
    "gemini-2.5-flash",          # More capable fallback
    "gemini-2.0-flash",          # Last resort
]

# === TRANSLATION SETTINGS ===
INPUT_LANGUAGE = "Japanese"
OUTPUT_LANGUAGE = "English"
ENABLE_OSB = True              # Outside speech bubble text detection
MAX_RETRIES_PER_IMAGE = 3      # Retry failed images this many times
RETRY_FAILED_AT_END = True     # Re-attempt all failed pages after first pass

print(f'✅ Config: {len(API_KEYS)} keys, {len(MODEL_CHAIN)} models in fallback chain')
print(f'   Models: {", ".join(MODEL_CHAIN)}')
print(f'   {INPUT_LANGUAGE} → {OUTPUT_LANGUAGE} | OSB: {"ON" if ENABLE_OSB else "OFF"}')

In [ ]:
# 4️⃣ Upload manga images (ZIP or folder)
import os, shutil, zipfile, tempfile
from pathlib import Path
from google.colab import files

INPUT_DIR = Path("./input_manga")
OUTPUT_DIR = Path("./output")
INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("Upload your manga images or a ZIP file:")
uploaded = files.upload()

for fname, data in uploaded.items():
    if fname.lower().endswith('.zip'):
        tmp = Path(tempfile.mktemp(suffix='.zip'))
        tmp.write_bytes(data)
        with zipfile.ZipFile(tmp) as zf:
            zf.extractall(INPUT_DIR)
        tmp.unlink()
        print(f'  📦 Extracted ZIP: {fname}')
    else:
        (INPUT_DIR / fname).write_bytes(data)
        print(f'  🖼️ Uploaded: {fname}')

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
all_images = sorted([f for f in INPUT_DIR.rglob('*') if f.suffix.lower() in IMAGE_EXTS])
print(f'\n✅ Found {len(all_images)} images to translate')

In [ ]:
# 5️⃣ Batch Translate with Key Rotation + Model Fallback + Auto-Retry
import time, re, sys
import torch
from pathlib import Path
from core.config import (
    CleaningConfig, DetectionConfig, MangaTranslatorConfig,
    OutputConfig, OutsideTextConfig, PreprocessingConfig,
    RenderingConfig, TranslationConfig,
)
from core.pipeline import translate_and_render
from core.validation import autodetect_yolo_model_path, clamp_settings

# ── Setup ──
models_dir = Path('./models').resolve()
models_dir.mkdir(parents=True, exist_ok=True)
yolo_path = str(autodetect_yolo_model_path(models_dir, 'yolo_1'))
font_dir = './fonts'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Device: {device}')

def build_config(api_key, model_name):
    config = MangaTranslatorConfig(
        yolo_model_path=yolo_path,
        verbose=False,
        device=device,
        parallel_requests=1,
        detection=DetectionConfig(
            confidence=0.6, conjoined_confidence=0.35, panel_confidence=0.25,
            seg_model='yolo', bubble_detector_model='yolo_1',
            conjoined_detection=True, use_panel_sorting=True,
            use_osb_text_verification=True,
        ),
        cleaning=CleaningConfig(thresholding_value=190, roi_shrink_px=5),
        translation=TranslationConfig(
            provider='Google', google_api_key=api_key, model_name=model_name,
            temperature=0.1, top_p=0.95, top_k=64,
            input_language=INPUT_LANGUAGE, output_language=OUTPUT_LANGUAGE,
            reading_direction='rtl', translation_mode='one-step',
            reasoning_effort='none', send_full_page_context=True,
            upscale_method='lanczos', context_image_max_side_pixels=768,
            bubble_min_side_pixels=128, ocr_method='LLM',
        ),
        rendering=RenderingConfig(font_dir=font_dir, max_font_size=16, min_font_size=8, supersampling_factor=2),
        output=OutputConfig(output_format='jpeg', jpeg_quality=92),
        outside_text=OutsideTextConfig(enabled=ENABLE_OSB, inpainting_method='opencv', osb_confidence=0.6, osb_outline_width=3.0),
        preprocessing=PreprocessingConfig(enabled=False, auto_scale=True),
    )
    clamp_settings(config)
    return config

RATE_LIMIT_KEYWORDS = ['429', 'Rate limit', 'RESOURCE_EXHAUSTED', 'Resource Exhausted', 'quota']
MODEL_ERROR_KEYWORDS = ['not found', 'does not exist', 'is not available', 'unsupported model', 'invalid model']

def is_rate_limit(err):
    return any(kw in str(err) for kw in RATE_LIMIT_KEYWORDS)

def is_model_error(err):
    return any(kw.lower() in str(err).lower() for kw in MODEL_ERROR_KEYWORDS)

# ── Collect images ──
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
images = sorted([f for f in INPUT_DIR.rglob('*') if f.suffix.lower() in IMAGE_EXTS])

# ── State ──
key_idx = 0
model_idx = 0
exhausted_keys = set()          # Keys that hit rate limits
dead_models = set()             # Models that don't exist/work
failed_images = []              # (img_path, out_path, error) for retry
success = 0
skipped = 0
times = []
total_start = time.time()

def get_current_key():
    """Get next available key, cycling through non-exhausted ones."""
    global key_idx
    if not API_KEYS:
        return None
    for _ in range(len(API_KEYS)):
        k = key_idx % len(API_KEYS)
        if k not in exhausted_keys:
            return API_KEYS[k]
        key_idx += 1
    return None  # All exhausted

def get_current_model():
    """Get next available model from the fallback chain."""
    for i in range(model_idx, len(MODEL_CHAIN)):
        if MODEL_CHAIN[i] not in dead_models:
            return MODEL_CHAIN[i]
    return None  # All dead

def advance_key():
    """Rotate to next key."""
    global key_idx
    exhausted_keys.add(key_idx % len(API_KEYS))
    key_idx += 1

def advance_model():
    """Fall to next model in chain."""
    global model_idx, exhausted_keys
    current = get_current_model()
    if current:
        dead_models.add(current)
    model_idx += 1
    exhausted_keys.clear()  # Reset key exhaustion for new model

def translate_one(img_path, out_path):
    """Try to translate one image with full rotation/fallback logic. Returns (success, error_msg)."""
    global key_idx
    for attempt in range(MAX_RETRIES_PER_IMAGE * len(MODEL_CHAIN)):
        model = get_current_model()
        key = get_current_key()
        if not model:
            return False, 'All models exhausted'
        if not key:
            # All keys exhausted for this model, try next model
            old_model = model
            advance_model()
            model = get_current_model()
            if not model:
                return False, f'All keys exhausted for {old_model}, no more models'
            key = get_current_key()
            if not key:
                return False, 'No available keys'
            print(f'\n       ↳ Model fallback: {old_model} → {model}')

        try:
            config = build_config(key, model)
            translate_and_render(img_path, config, out_path)
            return True, None
        except Exception as e:
            err = str(e)
            if is_rate_limit(err):
                k_label = (key_idx % len(API_KEYS)) + 1
                print(f' ⚠️ Key#{k_label} rate limited', end='', flush=True)
                advance_key()
                continue
            elif is_model_error(err):
                print(f' ⚠️ {model} unavailable', end='', flush=True)
                advance_model()
                continue
            else:
                return False, err[:200]
    return False, 'Max retries exceeded'

# ── Main translation loop ──
print('=' * 60)
print('🚀 MangaTranslator — Colab Batch Processor')
print('=' * 60)
print(f'  🔑 API Keys:  {len(API_KEYS)} loaded')
print(f'  🧠 Models:   {", ".join(MODEL_CHAIN)}')
print(f'  🖼️ Images:   {len(images)}')
print(f'  🚀 OSB:      {"ON" if ENABLE_OSB else "OFF"} | {INPUT_LANGUAGE} → {OUTPUT_LANGUAGE}')
print('-' * 60)

for i, img_path in enumerate(images):
    rel = img_path.relative_to(INPUT_DIR)
    out_path = OUTPUT_DIR / rel.with_suffix('.jpg')
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Skip already translated
    if out_path.exists():
        skipped += 1
        print(f'  [{i+1}/{len(images)}] ⏭️  SKIP: {rel}')
        continue

    # ETA
    eta = time.strftime('%M:%S', time.gmtime((sum(times)/len(times)) * (len(images)-i))) if times else '...'
    model = get_current_model() or '?'
    k_label = (key_idx % len(API_KEYS)) + 1 if API_KEYS else 0
    print(f'  [{i+1}/{len(images)}] 🔑#{k_label} {model} | {rel} (ETA: {eta})', end='', flush=True)

    t0 = time.time()
    ok, err = translate_one(img_path, out_path)
    elapsed = time.time() - t0

    if ok:
        times.append(elapsed)
        success += 1
        print(f' ✅ {elapsed:.1f}s')
    else:
        failed_images.append((img_path, out_path, err))
        print(f' ❌ {elapsed:.1f}s — {err[:100]}')

# ── Retry failed images ──
if RETRY_FAILED_AT_END and failed_images:
    print(f'\n{"=" * 60}')
    print(f'🔁 RETRYING {len(failed_images)} FAILED IMAGES')
    print(f'{"=" * 60}')
    # Reset exhaustion state for retry pass
    exhausted_keys.clear()
    dead_models.clear()
    model_idx = 0
    key_idx = 0
    still_failed = []
    for j, (img_path, out_path, prev_err) in enumerate(failed_images):
        if out_path.exists():
            continue  # Already done by a previous retry
        rel = img_path.relative_to(INPUT_DIR)
        model = get_current_model() or '?'
        print(f'  [retry {j+1}/{len(failed_images)}] {model} | {rel}', end='', flush=True)
        t0 = time.time()
        ok, err = translate_one(img_path, out_path)
        elapsed = time.time() - t0
        if ok:
            success += 1
            times.append(elapsed)
            print(f' ✅ {elapsed:.1f}s')
        else:
            still_failed.append((img_path, out_path, err))
            print(f' ❌ {elapsed:.1f}s — {err[:100]}')
    failed_images = still_failed

# ── Summary ──
total_time = time.time() - total_start
avg = sum(times) / len(times) if times else 0
print(f'\n{"=" * 60}')
print('📊 BATCH COMPLETE')
print(f'{"=" * 60}')
print(f'  Total:    {len(images)} images')
print(f'  ✅ Done:   {success}')
print(f'  ⏭️  Skip:   {skipped}')
print(f'  ❌ Failed: {len(failed_images)}')
print(f'  ⏱️ Time:   {time.strftime("%H:%M:%S", time.gmtime(total_time))}')
if times:
    print(f'  📈 Avg:    {avg:.1f}s/image')
print(f'  📁 Output: {OUTPUT_DIR.resolve()}')
print(f'{"=" * 60}')
if failed_images:
    print('\n⚠️  Still failed:')
    for img, _, err in failed_images:
        print(f'  • {img.name}: {err[:80]}')

In [ ]:
# 6️⃣ Download — CBZ (manga reader) or ZIP
import zipfile, re, os
from pathlib import Path
from xml.etree.ElementTree import Element, SubElement, tostring
from google.colab import files

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

def collect_output_images(out_dir):
    imgs = [f for f in Path(out_dir).rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
    imgs.sort(key=lambda x: tuple(int(p) if p.isdigit() else p for p in re.split(r'(\d+)', x.stem)))
    return imgs

def make_comic_info(title, page_count):
    root = Element('ComicInfo')
    root.set('xmlns:xsi', 'http://www.w3.org/2001/XMLSchema-instance')
    root.set('xmlns:xsd', 'http://www.w3.org/2001/XMLSchema')
    SubElement(root, 'Title').text = title
    SubElement(root, 'PageCount').text = str(page_count)
    SubElement(root, 'LanguageISO').text = 'en'
    SubElement(root, 'Manga').text = 'YesAndRightToLeft'
    SubElement(root, 'Notes').text = 'Translated by MangaTranslator'
    return b'<?xml version="1.0" encoding="utf-8"?>\n' + tostring(root, encoding='unicode').encode('utf-8')

out_images = collect_output_images(OUTPUT_DIR)
if not out_images:
    print('❌ No translated images found!')
else:
    title = 'MangaTranslator_output'
    # === CBZ (for manga readers: Tachiyomi, Mihon, Kavita, Komga, etc.) ===
    cbz_path = f'/tmp/{title}.cbz'
    with zipfile.ZipFile(cbz_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.writestr('ComicInfo.xml', make_comic_info(title, len(out_images)))
        pad = max(3, len(str(len(out_images))))
        for idx, img in enumerate(out_images, 1):
            zf.write(img, f'{str(idx).zfill(pad)}{img.suffix.lower()}')
    print(f'📦 CBZ ready: {len(out_images)} pages ({os.path.getsize(cbz_path)/1024/1024:.1f} MB)')

    # === ZIP (raw files) ===
    zip_path = f'/tmp/{title}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for img in out_images:
            zf.write(img, str(img.relative_to(OUTPUT_DIR)))
    print(f'📁 ZIP ready: {os.path.getsize(zip_path)/1024/1024:.1f} MB')

    print('\n⬇️ Choose download format:')
    choice = input('  [1] CBZ (manga readers)  [2] ZIP (raw files)  [3] Both : ').strip()
    if choice in ('1', '3'):
        files.download(cbz_path)
    if choice in ('2', '3'):
        files.download(zip_path)
    if choice not in ('1', '2', '3'):
        files.download(cbz_path)
        print('(Defaulted to CBZ)')

In [ ]:
# 7️⃣ (Optional) Launch Web UI with public link
!sed -i 's/show_error=True/show_error=True, share=True/' app.py 2>/dev/null
print('Starting MangaTranslator Web UI...')
print('Wait for the public URL below \u2193')
!python app.py